In [23]:
from datasets import load_dataset
dataset = load_dataset("Skylion007/openwebtext", split="train", streaming=True)

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

In [24]:
from datasets import IterableDataset

dataset = dataset.take(100_000)

In [25]:
with open('data.txt', 'w') as file:
  for i, example in enumerate(dataset):
    file.write(example['text'] + '\n')

In [26]:
import pandas as pd

df = pd.DataFrame(dataset)

del dataset

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

tokenizer = Tokenizer(BPE())

tokenizer.pre_tokenizer = ByteLevel()

trainer = BpeTrainer(
    vocab_size=50257,
    special_tokens=["<|endoftext|>"],
    initial_alphabet=ByteLevel.alphabet()
)

tokenizer.decoder = ByteLevelDecoder()

tokenizer.train(
    files=["data.txt"],
    trainer=trainer
)

tokenizer.save("tokenizer.json")

In [ ]:
import torch
from torch.utils.data import Dataset

class GPT2Dataset(Dataset):
    def __init__(self, df, tokenizer, block_size=512):
        super().__init__()
        self.block_size = block_size
        self.tokenizer = tokenizer
        self.df = df
        self.pad_id = tokenizer.token_to_id('<|endoftext|>')

    def __pad(self, ids):
        ids = ids[:self.block_size]
        ids = ids + [self.pad_id] * (self.block_size - len(ids))
        return ids

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df['text'][idx]
        encoding = self.tokenizer.encode(text)
        ids = self.__pad(encoding.ids)

        input_ids = torch.tensor(ids[:-1], dtype=torch.long)
        target_ids = torch.tensor(ids[1:], dtype=torch.long)
        target_ids[target_ids == self.pad_id] = -100

        return input_ids, target_ids

In [ ]:
VOCAB_SIZE = tokenizer.get_vocab_size()

In [ ]:
import torch.nn as nn
import torch

class embeddingGPT(nn.Module):
  def __init__(self,d_model=768,max_len = 512,vocab_size= VOCAB_SIZE):
    super().__init__()
    self.embedding = nn.Embedding(num_embeddings=vocab_size,embedding_dim=d_model)
    self.position = nn.Embedding(num_embeddings=max_len,embedding_dim=d_model)

  def forward(self, x):
    seq_len = x.size(1)
    position_val = torch.arange(seq_len,device=x.device).unsqueeze(0)

    input_ids = self.embedding(x)
    position = self.position(position_val)

    return input_ids + position


In [ ]:
class feedforward(nn.Module):
  def __init__(self):
    super().__init__()
    self.linear_layer1 = nn.Linear(in_features=1024, out_features=4096)
    self.Gelu = nn.GELU()
    self.linear_layer2 = nn.Linear(in_features=4096, out_features=1024)

  def forward_pass(self,x):
    return x + self.linear_layer2(self.Gelu(self.linear_layer1(x)))

In [ ]:
class Transformer(nn.Module):
  def __init__(self,d_model, num_heads):
    super().__init__()
    self.feedforward =  feedforward()

    self.masked_attn = nn.MultiheadAttention(embedd_dim=d_model, num_heads= num_heads)
    self.layer_norm1 = nn.LayerNorm()

    self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads)
    self.layer_norm2 = nn.LayerNorm()

    self.layer_norm3 = nn.LayerNorm()

  def forward_pass(self,x, attn_mask):

    masked_attn, _ = self.masked_attn(x,x,x,attn_mask=attn_mask)
    layer_norm = self.layer_norm1(masked_attn + x)

    attn_out, _ = self.attn(layer_norm,layer_norm,layer_norm)
    layer_norm2 = self.layer_norm2(attn_out + layer_norm)

    ff_layer = self.feedforward(layer_norm2)
    layer_norm = self.layer_norm3(layer_norm2 + ff_layer)

    return x + layer_norm

In [ ]:
import torch.nn as nn
import torch

class GPT2(nn.Module):
  def __init__(self,d_model,num_heads,max_len=512):
    super().__init__()

    self.embedding = embeddingGPT()

    self.blocks = nn.ModuleList([
                    Transformer(
                        d_model=d_model,
                        num_heads=num_heads
                    )
                    for _ in range(12)
                ])

    self.lm_head = nn.Linear(d_model, max_len)

  def forward(self,x,attn_mask):
    x = self.embedding(x)

    for block in self.blocks:
      ids = block(x, attn_mask)

    logits = self.lm_head(ids)

    return logits + x

In [ ]:
from torch.utils.data import DataLoader, random_split

dataset = GPT2Dataset(df, tokenizer)

length = len(dataset)

train_size = int(length * 0.8)
val_size = length - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=512 , shuffle = True,num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=512 , shuffle = False,num_workers=2)

In [ ]:
import torch.optim as optim

model = GPT2(d_model=1024, num_heads=16)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1
)